## Visão Geral do Microsoft Entra ID

Microsoft Entra ID (anteriormente Azure Active Directory) é o serviço de gerenciamento de identidade e acesso baseado em nuvem da Microsoft. Ele serve como provedor de identidade central para Microsoft 365, Azure e milhares de outras aplicações SaaS.

Recursos Principais:
* **Single Sign-On (SSO)** - Usuários se autenticam uma vez para acessar múltiplas aplicações
* **Multi-Factor Authentication (MFA)** - Segurança aprimorada através de métodos adicionais de verificação
* **Conditional Access** - Controle de acesso baseado em políticas com base em usuário, dispositivo, localização e risco
* **Application Integration** - Suporta protocolos de autenticação modernos como OAuth 2.0, OpenID Connect e SAML

## Objetivo de Aprendizado
Microsoft Entra ID pode ser usado como provedor de identidade no AgentCore Identity e usado para autenticar usuários e fazê-los autorizar o agente a acessar recursos protegidos em seu nome.

<img src="images/entra-notebook-overview.png" width="75%">

## Pré-requisitos

Para executar este tutorial você precisará de:

* Python 3.10+
* Credenciais AWS
* Strands Agents
* Docker, Finch ou Podman instalado
* Definir região AWS como "us-west-2" ou qualquer região que suporte Bedrock AgentCore. Consulte https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agentcore-regions.html para regiões suportadas.

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## Passo 1: Configurar Tenant do Entra ID

## Fluxo Authorization Code
O fluxo authorization code do OAuth 2.0 é a abordagem recomendada para aplicações web autenticarem usuários com segurança e obterem tokens de acesso. Este fluxo envolve:
1. Redirecionar usuários para o Entra ID para autenticação
2. Receber um código de autorização após login bem-sucedido
3. Trocar o código por tokens de acesso e refresh
4. Usar tokens para acessar recursos protegidos

Este padrão de integração permite que o AgentCore aproveite as robustas capacidades de gerenciamento de identidade do Entra ID enquanto mantém autenticação segura e baseada em padrões para suas aplicações.

Um tenant do Entra ID é uma instância dedicada do Microsoft Entra ID que representa sua organização. Pense nele como o diretório isolado da sua organização na nuvem da Microsoft.

Características Principais:
* **Identidade Única** - Cada tenant tem um domínio único (por exemplo, suaempresa.onmicrosoft.com)
* **Limite Isolado** - Usuários, grupos e aplicações em um tenant são separados de outros
* **Controle Administrativo** - Administradores do tenant gerenciam usuários, políticas de segurança e registros de aplicação
* **Suporte Multi-Domínio** - Pode incluir domínios personalizados além do domínio padrão .onmicrosoft.com

Na Prática:

Quando você registra uma aplicação com o Entra ID para integração OAuth, você está registrando-a dentro de um tenant específico. Usuários desse tenant podem então se autenticar em sua aplicação usando suas credenciais organizacionais.

Para integração com AgentCore, você precisará de:
* **Tenant ID** - Identificador único para a instância do Entra ID
* **Application Registration** - Sua aplicação registrada dentro do tenant
* **Permissões Apropriadas** - Direitos de acesso configurados para sua aplicação

Este modelo baseado em tenant garante que autenticação e autorização permaneçam dentro do limite de segurança da sua organização.

Passos para criar um tenant podem ser encontrados em https://learn.microsoft.com/en-us/entra/fundamentals/create-new-tenant

Nota:
1. Microsoft Entra ID não é um serviço AWS. Consulte a documentação do Microsoft Entra ID para detalhes relacionados a custos.
2. Capturas de tela usadas nos seguintes passos podem mudar. Encorajamos você a consultar a documentação do Microsoft Entra ID para orientações mais recentes sobre configuração de uma aplicação Entra ID.

## Passo 2: Configurar Aplicação

1. Vá para https://portal.azure.com e pesquise por "Entra ID" na barra de pesquisa no topo da tela
<img src="images/entraid.jpg" width="75%">

2. Vá para `Manage` &rarr; `App Registrations`
<img src="images/app.registration.png" width="75%">


3. Clique em `New Registration` e preencha os detalhes. Certifique-se de selecionar a opção multi tenant
<img src="images/app.registration.form.png" width="75%">


4. Crie um client secret. Copie o clientId e client secret para uso no AgentCore Identity.
<img src="images/gather.client.info.png" width="75%">


5. Crie Scopes para OAuth. Vá para Expose an API &rarr; `Add Scope`. Copie e salve o scope completo.
<img src="images/expose.api.png" width="75%">


6. Adicione permissões de API para permitir acesso ao OneNote. "API Permissions" --> "Add a permission" --> "Microsoft API" --> "OneNote" --> "Delegated Permissions"
<img src="images/onenote.api.perm.png" width="75%"/>

## Passo 2 - Criar um Provedor de Identidade Bedrock AgentCore

Atualize as variáveis de ambiente abaixo usando os detalhes do tenant e aplicação que você criou no Passo 1.

recuperar:
- Tenant ID de "App registration" --> "All Applications" --> Selecione o cliente que você acabou de criar --> "Overview" --> "Directory (tenant) ID"
- Client ID de "App registration" --> "All Applications" --> Selecione o cliente que você acabou de criar --> "Overview" --> "Application (client) ID"
- Secret salvo do passo anterior
- Scope será "openid profile https://graph.microsoft.com/Notes.ReadWrite.All https://graph.microsoft.com/Notes.Create"
- Audience será "https://graph.microsoft.com"

In [ ]:
import os

# REPLACE WITH YOUR client_id
os.environ["client_id"] = "REPLACE_ME"

# REPLACE WITH YOUR secret
os.environ["secret"] = "REPLACE_ME"

# REPLACE WITH YOUR tenant_id
os.environ["tenant_id"] = "REPLACE_ME"

##########

# REPLACE WITH YOUR scopes, if needed
os.environ["scopes"] = (
    "openid profile https://graph.microsoft.com/Notes.ReadWrite.All https://graph.microsoft.com/Notes.Create"
)

# REPLACE WITH YOUR audience, if needed
os.environ["audience"] = "https://graph.microsoft.com"

Amazon Bedrock AgentCore Identity fornece provedores suportados OAuth 2.0 gerenciados para autenticação de entrada e saída.

Crie um provedor de identidade para uso com seu agente. Um provedor abstrai a complexidade de diferentes implementações OAuth 2.0, esquemas de autenticação de API e formatos de token, apresentando uma interface unificada para agentes enquanto lida com variações de protocolo subjacentes e casos extremos.

In [ ]:
from bedrock_agentcore.services.identity import IdentityClient
from boto3.session import Session


boto_session = Session()
region = boto_session.region_name

if not region:
    import warnings

    warnings.warn(
        "There is no configured Region in the AWS session. Defaulting to us-east-1"
    )
    region = "us-east-1"

identity_client = IdentityClient(region=region)

ms_provider = identity_client.create_oauth2_credential_provider(
    req={
        "name": "microsoft_entra_oauth_provider",
        "credentialProviderVendor": "MicrosoftOauth2",
        "oauth2ProviderConfigInput": {
            "microsoftOauth2ProviderConfig": {
                "clientId": os.environ["client_id"],
                "clientSecret": os.environ["secret"],
                "tenantId": os.environ["tenant_id"],
            }
        },
    }
)
print(f"Microsoft Credential Provider: {ms_provider}")
print()
print(f"Callback URL: {ms_provider['callbackUrl']}")

## Passo 2.5: Atualizar Microsoft Entra com a URL de Callback do Provedor de Credenciais

<img src="images/redirect.uri.png" width="75%">

Selecione Token para emitir.

<img src="images/select.tokens.jpg" width="75%">

## Passo 3: Validar localmente

AgentCore Identity permite que desenvolvedores obtenham tokens OAuth para acesso delegado pelo usuário ou autenticação máquina-para-máquina com base nos provedores de credenciais OAuth 2.0 configurados.

O serviço orquestrará o processo de autenticação entre o usuário ou aplicação e o servidor de autorização downstream, e recuperará e armazenará o token resultante. Uma vez que o token esteja disponível no cofre do AgentCore Identity, agentes autorizados podem recuperá-lo e usá-lo para autorizar chamadas para servidores de recursos.

##### No código abaixo, estamos usando Entra ID para um fluxo delegado pelo usuário.

In [ ]:
from bedrock_agentcore.identity.auth import requires_access_token
from oauth2_callback_server import get_oauth2_callback_url

@requires_access_token(
    provider_name="microsoft_entra_oauth_provider",
    auth_flow="USER_FEDERATION",
    scopes=os.environ["scopes"].split(" "),
    on_auth_url=lambda x: print(
        "\nPlease copy and paste this URL in your browser:\n" + x
    ),
    force_authentication=True,
    callback_url=get_oauth2_callback_url(),
)
def need_access_token(*, access_token: str):
    return access_token

##### `need_access_token(access_token="")` apresentará uma URL que você usa para se autenticar no Entra ID e obter um token de autorização para a aplicação usar. Uma vez que você tenha se autenticado e compartilhado seu consentimento, o código de autorização estará disponível para você.

<img src="images/authenticate.and.authorize.png" width="75%">


In [ ]:
import sys
import subprocess

from oauth2_callback_server import wait_for_oauth2_server_to_be_ready

oauth2_callback_server_cmd = [
    sys.executable,
    "oauth2_callback_server.py",
    "--region",
    region,
]
oauth2_callback_server_process = subprocess.Popen(oauth2_callback_server_cmd)

id_token = ""
try:
    successfully_started_oauth2_server = wait_for_oauth2_server_to_be_ready()
    if not successfully_started_oauth2_server:
        print(
            "Failed to start OAuth2 callback server to handle session binding "
            "(https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/oauth2-authorization-url-session-binding.html)"
        )
    else:
        id_token = need_access_token(access_token="")
        print(f"Bearer Token Received: {id_token[:10]}...")
finally:
    oauth2_callback_server_process.terminate()

##### Você pode decodificar o token e validá-lo localmente.

In [ ]:
import json
import jwt  # PyJWT library

# Decode the token (without verification for inspection purposes only)
# For production, always verify the token's signature and claims
decoded_token = jwt.decode(id_token, options={"verify_signature": False})

print(f"Decoded Bearer Token (for inspection): \n{json.dumps(decoded_token, indent=4)}")


##### Seu token decodificado do Entra ID deve parecer similar ao abaixo.
<img src="images/decoded-token.png" width="75%">

## Passo 4 - Juntar tudo como um Agente AgentCore Runtime

### Agente de Integração OneNote

Este código cria um agente de IA que ajuda usuários a criar e gerenciar notebooks Microsoft OneNote através de comandos em linguagem natural. O agente usa autenticação EntraID para acessar a API do OneNote e fornece três funções principais:

1. Create Notebook - Cria um novo notebook OneNote (ferramenta `create_notebook`)
2. Create Section - Adiciona seções aos notebooks existentes (ferramenta `create_notebook_section`)
3. Add Content - Cria páginas com conteúdo em seções do notebook (ferramenta `add_content_to_notebook_section`)

O agente manipula autenticação OAuth2 automaticamente, solicitando que os usuários autorizem quando necessário, e então processa suas requisições para organizar notas de reuniões ou outro conteúdo em notebooks OneNote estruturados.

In [ ]:
%%writefile strands_entraid_onenote.py
import os
import json
import asyncio
import requests

from strands import Agent
from strands import tool
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from bedrock_agentcore.identity.auth import requires_access_token
from strands.models.bedrock import BedrockModel
from oauth2_callback_server import get_oauth2_callback_url

os.environ["STRANDS_OTEL_ENABLE_CONSOLE_EXPORT"] = "true"
os.environ["OTEL_PYTHON_EXCLUDED_URLS"] = "/ping,/invocations"

entra_access_token = None  # Global variable to store the access token
tool_name = None

@tool
def create_notebook(name: str) -> str:
    """
    Create a new Microsoft OneNote notebook for the user. Needed before you can create a section or add content.
    
    Args:
        name (str): The display name for the new notebook
        
    Returns:
        str: The ID of the created notebook
    """
    global entra_access_token
    global tool_name 
    tool_name = "create_notebook"
    # Check if we already have a token
    if not entra_access_token:
        return json.dumps({"auth_required": True, "message": f"Entra ID authentication is required for {tool_name}. Please wait while we set up the authorization.", "events": []})

    headers = {
        'Authorization': f'Bearer {entra_access_token}',
        'Content-Type': 'application/json'
    }
    # Create new notebook
    notebook_data = {'displayName': name}
    notebook = requests.post(
        'https://graph.microsoft.com/v1.0/me/onenote/notebooks', 
        headers=headers, 
        json=notebook_data
    )
    notebook.raise_for_status()
    return json.dumps({"notebook_id": notebook.json()['id']})

@tool
def create_notebook_section(notebook_id: str, section_name: str) -> str:
    """
    Create a new section in an existing OneNote notebook. Section is created for a specific notebook. 
    
    Args:
        notebook_id (str): The ID of the OneNote notebook to create the section in
        section_name (str): The display name for the new section
        
    Returns:
        str: The ID of the created section
    """
    global entra_access_token
    global tool_name 
    tool_name = "create_notebook_section"
    # Check if we already have a token
    if not entra_access_token:
        return json.dumps({"auth_required": True, "message": f"Entra ID authentication is required for {tool_name}. Please wait while we set up the authorization.", "events": []})

    headers = {
        'Authorization': f'Bearer {entra_access_token}',
        'Content-Type': 'application/json'
    }
    # Create new section
    section_data = {'displayName': section_name}
    section = requests.post(
        f'https://graph.microsoft.com/v1.0/me/onenote/notebooks/{notebook_id}/sections',
        headers=headers, 
        json=section_data
    )
    section.raise_for_status()
    
    section_id = section.json()['id']
    return json.dumps({"section_id": section_id})

@tool
def add_content_to_notebook_section(section_id: str, page_content) -> str:
    """
    Add content to a OneNote notebook section by creating a new page.
    
    Args:
        section_id (str): The ID of the OneNote section to add content to
        page_content: The HTML content to add as a new page
        
    Returns:
        str: URL to the created notebook page
    """
    global entra_access_token
    global tool_name 
    tool_name = "add_content_to_notebook_section"
    
    # Check if we already have a token
    if not entra_access_token:
        return json.dumps({"auth_required": True, "message": f"Entra ID authentication is required for {tool_name}. Please wait while we set up the authorization.", "events": []})

    headers = {
        'Authorization': f'Bearer {entra_access_token}',
        'Content-Type': 'text/html'
    }
    page = requests.post(
        f'https://graph.microsoft.com/v1.0/me/onenote/sections/{section_id}/pages',
        headers=headers, 
        data=page_content
    )
    page.raise_for_status()
    url = json.loads(page.text)["links"]["oneNoteWebUrl"]["href"]
    return json.dumps({"oneNoteWebUrl": url})
    
    
# Initialize the agent with tools
model = BedrockModel(model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0")
system_prompt = """You are an Agent who helps user put in their meeting into OneNote notebooks. 
    Identify the notebook name, section name and content based on what the user has provided. 
    Return notebook URL once created."""
agent = Agent(model=model, system_prompt=system_prompt, tools=[create_notebook, create_notebook_section, add_content_to_notebook_section])

# Initialize app and streaming queue
app = BedrockAgentCoreApp()

class StreamingQueue:
    def __init__(self):
        self.finished = False
        self.queue = asyncio.Queue()
        
    async def put(self, item):
        await self.queue.put(item)

    async def finish(self):
        self.finished = True
        await self.queue.put(None)

    async def stream(self):
        while True:
            item = await self.queue.get()
            if item is None and self.finished:
                break
            yield item

queue = StreamingQueue()

async def on_auth_url(url: str):
    print(f"Authorization url: {url}")
    await queue.put(f"Authorization url: {url}")


async def agent_task(user_message: str):
    global tool_name
    try:
        await queue.put("Begin agent execution")
        
        # Call the agent first to see if it needs authentication
        response = agent(user_message)
        
        # Extract text content from the response structure
        response_text = ""
        if isinstance(response.message, dict):
            content = response.message.get('content', [])
            if isinstance(content, list):
                for item in content:
                    if isinstance(item, dict) and 'text' in item:
                        response_text += item['text']
        else:
            response_text = str(response.message)
        
        # Check if the response indicates authentication is required
        # Look for various keywords that indicate authentication issues
        auth_keywords = [
            "authentication", "authorize", "authorization", "auth", 
            "sign in", "login", "access", "permission", "credential",
            "need authentication", "requires authentication"
        ]
        needs_auth = any(keyword.lower() in response_text.lower() for keyword in auth_keywords)
       
        if needs_auth:
            await queue.put(f"Authentication required for {tool_name} access. Starting authorization flow...")
            
            # Trigger the 3LO authentication flow
            try:
                global entra_access_token
                entra_access_token = await need_token_3LO_async(access_token=None)
                await queue.put(f"Authentication successful! Retrying {tool_name}...")
                
                # Retry the agent call now that we have authentication
                response = agent(user_message)
            except Exception as auth_error:
                print(f"auth_error: ", repr(auth_error))
                await queue.put(f"Authentication failed: {repr(auth_error)}")
        
        await queue.put(response.message)
        await queue.put("End agent execution")
    except Exception as e:
        await queue.put(f"Error: {repr(e)}")
    finally:
        await queue.finish()

@requires_access_token(
    provider_name="microsoft_entra_oauth_provider",
    scopes=os.environ["scopes"].split(' '),
    auth_flow='USER_FEDERATION',
    on_auth_url=on_auth_url,
    force_authentication=True,
    callback_url=get_oauth2_callback_url(),
)
async def need_token_3LO_async(*, access_token: str):
    global entra_access_token
    entra_access_token = access_token  # Update the global access token
    print("Got access token....", access_token)
    return access_token


@app.entrypoint
async def agent_invocation(payload):
    user_message = payload.get("prompt", "No prompt found in input, please guide customer to create a json payload with prompt key")
    
    # Create and start the agent task
    task = asyncio.create_task(agent_task(user_message))

    # Return the stream, but ensure the task runs concurrently
    async def stream_with_task():
        # Stream results as they come
        async for item in queue.stream():
            yield item
        
        # Ensure the task completes
        await task
    
    return stream_with_task()
    
if __name__ == "__main__":
    app.run()


##### Configure seu AgentCore Runtime

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="strands_entraid_onenote.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="strands_entraid_onenote_3lo",
)

print(f"Agent Configure Response: {response}")

##### Lance seu Agente. Uma vez lançado, o agente estará disponível para uso em sua aplicação

In [ ]:
launch_response = agentcore_runtime.launch(
    local_build=False,
    auto_update_on_conflict=True,
    env_vars={
        "scopes": os.environ["scopes"],
    },
)

print(f"Launch Response: {launch_response}")

#### A célula de código abaixo fornecerá uma URL. Copie a URL que você obtiver (não a da imagem abaixo), para se autenticar usando janela do navegador.
<img src="images/url.presented.png" width="75%">

#### Você será solicitado a se autenticar. Complete a autenticação.
<img src="images/authenticate.and.authorize.png" width="75%">

#### Certifique-se de deletar o notebook com nome "Bedrock Agents" se ele já existir.

In [ ]:
import sys
import uuid
import subprocess

from typing import Final
from oauth2_callback_server import (
    store_user_id_in_oauth2_callback_server,
    wait_for_oauth2_server_to_be_ready,
)


prompt = """
Put these notes into onenote notebook named "Bedrock Agents".

Amazon Bedrock AgentCore enables you to deploy and operate 
highly capable AI agents securely, at scale. It offers 
infrastructure purpose-built for dynamic agent workloads, 
powerful tools to enhance agents, and essential controls for 
real-world deployment. AgentCore services can be used 
 together or independently and work with any framework including 
CrewAI, LangGraph, LlamaIndex, and Strands Agents, as well as 
any foundation model in or outside of Amazon Bedrock, giving you 
ultimate flexibility. AgentCore eliminates the undifferentiated 
heavy lifting of building specialized agent infrastructure, so 
you can accelerate agents to production. Provide link to 
the created OneNote Notebook and provide error message from the API 
in case of failure.
"""
session_id = str(uuid.uuid1())
user_id: Final[str] = "user"

oauth2_callback_server_cmd = [
    sys.executable,
    "oauth2_callback_server.py",
    "--region",
    region,
]
oauth2_callback_server_process = subprocess.Popen(oauth2_callback_server_cmd)

try:
    successfully_started_oauth2_server = wait_for_oauth2_server_to_be_ready()
    if not successfully_started_oauth2_server:
        print(
            "Failed to start OAuth2 callback server to handle session binding "
            "(https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/oauth2-authorization-url-session-binding.html)"
        )
    else:
        store_user_id_in_oauth2_callback_server(user_id)
        st = agentcore_runtime.invoke(
            payload={"prompt": prompt}, session_id=session_id, user_id=user_id
        )
finally:
    oauth2_callback_server_process.terminate()

## Passo 4 - Validar o notebook OneNote criado
- A função de invocação do agente acima criará um Notebook, uma Seção dentro dele, e adicionará algum conteúdo à seção.
- O usuário pode acessar o notebook criado fazendo login em https://seu-nome-de-dominio-my.sharepoint.com/

Você pode obter o nome de domínio de

<img src="images/domain.name.png" width="75%"/>

Após fazer login você chegará à sua página inicial do sharepoint.

<img src="images/sharepoint.home.png" width="75%"/>

De lá navegue para My Files --> Notebooks

<img src="images/sharepoint.my.files.png" width="75%"/>

## Conclusão e limpeza

Neste notebook aprendemos como:
- Configurar API e Aplicação do Entra ID para fornecer fluxo OAuth Authorization Code
- Criar um AgentCore Runtime e implantar agente com ferramentas que agiram em nome do usuário para criar notebooks OneNote

#### Recurso(s) criado(s)

In [ ]:
print(f"Runtime Agent: {launch_response.agent_id}")

#### Deletar AgentCore Runtime

In [ ]:
import os
import boto3

agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)

delete_agent_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_response.agent_id
)
print(delete_agent_response)
print()

try:
    os.remove(".bedrock_agentcore.yaml")
    print("Successfully deleted local Runtime Agent config")
except Exception as e:
    print(f"Failed to delete local Agent config: {repr(e)}")

#### Deletar provedor de credenciais OAuth2

In [ ]:
delete_credential_provider = agentcore_control_client.delete_oauth2_credential_provider(
    name=ms_provider["name"]
)
print(delete_credential_provider)